<a href="https://colab.research.google.com/github/Binamra00/rs-replication/blob/main/rel_tag_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⛏️ Architectural Inflection Point Extraction Pipeline
**Project:** Smell Ranker: Prioritizing Technical Debt across Architectural Eras  

This notebook houses the data extraction pipeline for mapping the longitudinal evolution of technical debt. The scripts below do not merely clone repositories; they programmatically distill years of software history into a scientifically rigorous timeline of **Architectural Boundaries** across six distinct software archetypes.

---

## 🏛️ The Methodological Framework: The "Goldilocks Zone"
In empirical software repository mining, analyzing every single Git commit or patch release generates massive statistical noise. According to the Semantic Versioning (SemVer) standard, patch releases (`X.Y.Z`) are strictly for backwards-compatible bug fixes. Furthermore, empirical research (Tufano et al., 2015) proves that code smells and structural technical debt are predominantly introduced during the implementation of new features, not during routine bug-fixing.

To achieve a high signal-to-noise ratio, this pipeline targets the **Major/Minor Era** (the "Goldilocks Zone"). By extracting only the **Genesis Point** (the `.0` release introducing a new architecture) and the **Terminal Point** (the final, most mature patch of that era) for each version cycle, we can accurately measure the "Debt Delta"—the exact accumulation or resolution of code smells between structural shifts (Chatzigeorgiou & Manakos, 2014; AlOmar et al., 2018).

---

## 🧬 The Generalization Dataset (Archetype Matrix)
To prove that the *Smell Ranker* tool is universally applicable, this pipeline extracts data from six highly diverse, enterprise-scale repositories, purposefully selected to represent fundamentally different domains of software engineering:

1. **Apache Commons Lang** *(Utility Library)*: A foundational, highly stable library.
2. **JUnit 5** *(Testing Framework)*: The industry-standard testing backbone.
3. **Spring Framework** *(Enterprise Web Framework)*: A massive, long-lifecycle IoC container.
4. **Apache Dubbo** *(Microservices / RPC)*: A highly distributed, network-heavy framework.
5. **QuestDB** *(High-Performance Database)*: A zero-GC, hardware-optimized time-series database.
6. **Checkstyle** *(Static Analysis Tool)*: An AST-parsing tool providing a meta-analytical layer to our own static analysis.

---

## ⚙️ Pipeline Execution Phases
Each repository extraction script below executes a strictly standardized, three-phase protocol tailored to bypass the unique historical tagging quirks (e.g., SVN migrations, missing digits, dormant periods) of that specific project:

* **Phase 1: Raw Tag Audit:** Clones the repository and extracts the entire unedited tagging history.
* **Phase 2: Clean GA Filtering:** Applies strict, repository-specific Regular Expression allowlists to eradicate pre-releases, milestones (`-M`), release candidates (`RC`), and betas.
* **Phase 3: Inflection Mapping & JSON Generation:** Groups the surviving releases into their architectural eras, isolates the Genesis and Terminal points, and generates a strict, pipeline-ready JSON manifest for static analysis.

***Ready to begin extraction. Run the cells below sequentially.***

### ⛏️ Repository Tag Extraction & Validation: Apache Commons Lang
**Archetype:** Utility Library

This script establishes the longitudinal release history for **Apache Commons Lang**. Because empirical software engineering requires precise historical snapshots, we cannot rely on arbitrary commits. We must anchor our static analysis to officially recognized General Availability (GA) releases.

**The Repository Quirk: The "RC Promotion" Standard**
Unlike modern repositories that create a clean `vX.Y.Z` tag for a final release, the Apache Commons team historically utilized a Release Candidate (RC) promotion model. They would tag `RC1`, `RC2`, etc., and once a candidate passed the Apache voting process, that specific `RC` tag was published as the official GA release *without being renamed*.

To accurately map this repository's architectural evolution while ensuring absolute data integrity, this script performs a four-phase extraction:
* **Phase 1: Raw Tag Audit:** Clones the repository, normalizes legacy tag formats (e.g., converting `_` to `.`), and audits the raw tags.
* **Phase 2: Genesis & Terminal Extraction:** Resolves the highest RC tag for each official release, groups them into `Major.Minor` architectural eras, and extracts the bounding Genesis and Terminal release candidates.
* **Phase 3: Official Release Mapping:** A strict validation matrix that cross-references every extracted boundary point against the 35 verified Apache target releases to ensure zero non-official tags leaked into the dataset.
* **Phase 4: JSON Manifest Generation:** Outputs the final pipeline-ready JSON file.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/apache/commons-lang.git"
REPO_DIR = "commons-lang"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
candidates = defaultdict(list)
major_count = minor_count = patch_count = non_version_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # Normalize legacy tags (LANG_3_1 -> LANG.3.1)
    normalized_tag = tag.replace('_', '.')
    match = re.search(r'(\d+)\.(\d+)(?:\.(\d+))?', normalized_tag)

    if match:
        major = int(match.group(1))
        minor = int(match.group(2))
        patch = int(match.group(3)) if match.group(3) else 0

        if minor == 0 and patch == 0: major_count += 1
        elif patch == 0: minor_count += 1
        else: patch_count += 1
    else:
        non_version_count += 1

    # FILTER: Ignore early testing noise
    if any(x in tag.upper() for x in ['BETA', 'DEV', 'STRUTS', 'COPY', 'REPACKAGED', '_B']):
        continue

    v_match = re.search(r'(\d+\.\d+(?:\.\d+)?)', normalized_tag)
    if not v_match: continue
    base_v = v_match.group(1)

    # [REVIEWER FIX]: Prioritize true GA releases over pre-release RCs
    is_rel = tag.startswith("rel/")
    rc_match = re.search(r'RC(\d+)', tag.upper())

    # Assign a massive weight (999) to 'rel/' tags so they always win the sort.
    # Otherwise, parse the RC number, or default to 0 for un-numbered tags.
    rc_num = 999 if is_rel else (int(rc_match.group(1)) if rc_match else 0)

    candidates[base_v].append({
        'tag': tag,
        'rc': rc_num,
        'date': date[:10],
        'sha': true_sha[:12],
        'major': major,
        'minor': minor,
        'patch': patch,
        'era_key': f"{major}.{minor}"
    })

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags[-50:]:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")
print("... (truncated for brevity) ...")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print(f"Major Releases Found: {major_count}")
print(f"Minor Releases Found: {minor_count}")
print(f"Patch Releases Found: {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: GENESIS & TERMINAL EXTRACTION
# ==========================================
official_releases = [
    "3.20.0", "3.19.0", "3.18.0", "3.17.0", "3.16.0", "3.15.0", "3.14.0", "3.13.0", "3.12.0",
    "3.11", "3.10", "3.9", "3.8.1", "3.8", "3.7", "3.6", "3.5", "3.4", "3.3.2", "3.3.1", "3.3",
    "3.2.1", "3.2", "3.1", "3.0.1", "3.0", "2.6", "2.5", "2.4", "2.3", "2.2", "2.1", "2.0",
    "1.0.1", "1.0"
]

clean_ga_tags = []
for v in official_releases:
    search_versions = [v, f"{v}.0", v.replace(".0", "")]
    found_cands = []

    for sv in search_versions:
        if sv in candidates:
            found_cands.extend(candidates[sv])

    if found_cands:
        sorted_cands = sorted(found_cands, key=lambda x: x['rc'], reverse=True)
        best = sorted_cands[0]
        best['official_target'] = v # Save this for Phase 3 validation
        clean_ga_tags.append(best)

pre_release_count = len(all_tags) - non_version_count - len(clean_ga_tags)

print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 70 + "\n")

eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# ==========================================
# PHASE 3: OFFICIAL RELEASE MAPPING & TRANSPARENCY
# ==========================================
print("--- 🛡️ Phase 3: Genesis/Terminal Transparency Matrix ---")
print(f"{'Target Version':<15} | {'Extracted Tag':<25} | {'Methodological Status'}")
print("-" * 80)

verified_count = 0
discarded_count = 0
missing_count = 0

# Create quick lookup sets
target_to_clean_ga = {obj['official_target']: obj for obj in clean_ga_tags}
sampled_tag_names = {obj['tag'] for obj in sampled_tags_reversed}

for target in official_releases:
    if target in target_to_clean_ga:
        ga_obj = target_to_clean_ga[target]
        if ga_obj['tag'] in sampled_tag_names:
            # Determine if it's a Genesis or Terminal point
            era = ga_obj['era_key']
            if ga_obj['tag'] == eras[era][0]['tag']:
                status = "✅ VALID (Genesis Point)"
            else:
                status = "✅ VALID (Terminal Point)"
            verified_count += 1
            print(f"{target:<15} | {ga_obj['tag']:<25} | {status}")
        else:
            status = "⏭️ DISCARDED (Middle Patch Noise)"
            discarded_count += 1
            print(f"{target:<15} | {ga_obj['tag']:<25} | {status}")
    else:
        status = "❌ MISSING (Not Found in Repo)"
        missing_count += 1
        print(f"{target:<15} | {'[NONE]':<25} | {status}")

print("-" * 80)
print(f"Transparency Report: {verified_count} Boundaries Validated, {discarded_count} Middle Patches Discarded, {missing_count} Missing.")
print("="*80 + "\n")

# ==========================================
# PHASE 4: FINAL JSON MANIFEST
# ==========================================
print("--- 🏁 Phase 4: Final JSON Manifest for commons-lang.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_obj['tag']}\"{comma}")
print("  ]\n}")

### ⛏️ Repository Tag Extraction & Validation: JUnit 5
**Archetype:** Testing Framework

This script maps the evolutionary timeline of **JUnit 5**, the industry-standard testing framework for the Java ecosystem.

**The Repository Quirk: The 'r' Prefix & SemVer Milestones**
Unlike Apache Commons, the JUnit team adheres much closer to modern Semantic Versioning (SemVer) but employs a specific prefix: all official JUnit 5 (and the upcoming JUnit 6) tags begin with `r` (e.g., `r5.8.2`). Furthermore, their development lifecycle relies heavily on public milestones and release candidates tagged with hyphens (e.g., `r5.0.0-M1`, `r5.0.0-RC2`).

To isolate the true architectural inflection points from the preliminary testing noise, this script executes a structured three-phase extraction:
* **Phase 1: Raw Tag Audit:** Clones the repository, extracts all tags, and parses them to calculate the exact distribution of Major, Minor, and Patch releases across the project's history.
* **Phase 2: Genesis & Terminal Extraction (Clean GA):** Applies a strict filter to isolate only `r5.*` and `r6.*` tags while programmatically stripping out any tag containing a hyphen (`-`). It then groups these pure General Availability (GA) releases into their `Major.Minor` eras to extract the bounding Genesis and Terminal points.
* **Phase 3: JSON Manifest Generation:** Outputs the final pipeline-ready JSON file.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/junit-team/junit-framework.git"
REPO_DIR = "junit5"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # Analyze SemVer for Audit Metrics
    match = re.search(r'(\d+)\.(\d+)\.(\d+)', tag)
    if match:
        maj, min, pat = match.groups()
        if min == '0' and pat == '0': major_count += 1
        elif pat == '0': minor_count += 1
        else: patch_count += 1

    # --- Strict Allowlist Filter Logic ---
    # FILTER 1: Only look at JUnit 5 and the new JUnit 6 tags
    if not (tag.startswith('r5.') or tag.startswith('r6.')):
        non_version_count += 1
        continue

    # FILTER 2: Exclude all pre-releases (Milestones, RCs, Alphas)
    if '-' in tag:
        pre_release_count += 1
        continue

    # If it survives, parse it into an era object
    clean_match = re.match(r'^r(\d+)\.(\d+)\.(\d+)$', tag)
    if clean_match:
        major = int(clean_match.group(1))
        minor = int(clean_match.group(2))
        patch = int(clean_match.group(3))

        clean_ga_tags.append({
            'tag': tag,
            'date': date[:10],
            'sha': true_sha[:12],
            'era_key': f"{major}.{minor}",
            'major': major,
            'minor': minor,
            'patch': patch
        })

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<25} | {'Release Date':<15} | {'SHA'}")
print("-" * 65)
for t in all_tags: # Truncated to 50 for Colab readability
    print(f"{t['tag']:<25} | {t['date']:<15} | {t['sha']}")


print("\n" + "="*65)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print(f"Major Releases Found: {major_count}")
print(f"Minor Releases Found: {minor_count}")
print(f"Patch Releases Found: {patch_count}")
print("="*65 + "\n")


# ==========================================
# PHASE 2: GENESIS & TERMINAL EXTRACTION
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 65)

print("\n--- 🗺️ Phase 2: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    role = "Genesis Point (.0)" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_obj['tag']:<25} | {role:<20} | {tag_obj['date']}")
print("\n")


# ==========================================
# PHASE 3: FINAL JSON MANIFEST
# ==========================================
print("--- 🏁 Phase 3: Final JSON Manifest for junit5.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_obj['tag']}\"{comma}")
print("  ]\n}")

### ⛏️ Repository Tag Extraction & Validation: Spring Framework
**Archetype:** Enterprise Web Framework

This script extracts and maps the evolutionary timeline of the **Spring Framework**. As a massive enterprise framework, Spring's architectural shifts occur over long lifecycles, meaning patch-by-patch analysis would generate excessive "maintenance noise" rather than capturing true structural technical debt.

**The Repository Quirk: Legacy vs. Modern Tagging & Era Boundaries**
Spring's tagging history is split into two eras. Older releases used a legacy suffix format (`vX.Y.Z.RELEASE`), while modern releases (v5.3+) migrated to strict Semantic Versioning (`vX.Y.Z`). Furthermore, because Spring's Major versions last for years, meaningful architectural shifts occur at the Minor version level (e.g., v4.1, v4.2, v4.3).

To capture these shifts precisely, this script utilizes a three-phase extraction:
* **Phase 1 & 2: Strict Allowlist Filtering:** Extracts all tags and uses dual Regular Expressions to isolate only pure General Availability (GA) releases, completely eradicating Milestones (`-M`) and Release Candidates (`-RC`).
* **Phase 3: Inflection Threshold Mapping (The "Goldilocks" Zone):** Groups the clean GA releases into their `Major.Minor` architectural eras. It then extracts only the "Genesis Point" (the `.0` release introducing the new architecture) and the "Terminal Point" (the final patch representing the mature architecture) of each era. This isolates the exact "Debt Delta" boundaries for static analysis.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/spring-projects/spring-framework.git"
REPO_DIR = "spring-framework"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12]
    })

    # --- Strict Allowlist Filter Logic ---
    if not tag.startswith('v'):
        non_version_count += 1
        continue

    is_clean_ga = False

    # Allow 1: Modern SemVer format exactly (e.g., v6.1.0)
    if re.match(r'^v\d+\.\d+\.\d+$', tag):
        is_clean_ga = True

    # Allow 2: Legacy Spring format exactly (e.g., v4.3.5.RELEASE)
    elif re.match(r'^v\d+\.\d+\.\d+\.RELEASE$', tag):
        is_clean_ga = True

    if not is_clean_ga:
        pre_release_count += 1
        continue

    # If it survives, it is a 100% pure GA Release
    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12]
    })

    # Semantic Versioning Breakdown
    match = re.search(r'^v(\d+)\.(\d+)\.(\d+)', tag)
    if match:
        major = int(match.group(1))
        minor = int(match.group(2))
        patch = int(match.group(3))

        if minor == 0 and patch == 0: major_count += 1
        elif patch == 0: minor_count += 1
        else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<25} | {'Creation Date':<15} | {'SHA'}")
print("-" * 65)
for t in all_tags:
    print(f"{t['tag']:<25} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*65)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*65 + "\n")


# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print(f"{'Clean GA Release':<25} | {'Release Date':<15} | {'SHA'}")
print("-" * 65)
for t in clean_ga_tags:
    print(f"{t['tag']:<25} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*65)
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*65 + "\n")


# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)

for tag_obj in clean_ga_tags:
    tag = tag_obj['tag']
    # Extract the X.Y part of vX.Y.Z
    match = re.search(r'v(\d+\.\d+)', tag)
    if match:
        era_key = match.group(1)
        eras[era_key].append(tag_obj)

sampled_tags = []

# Sort eras chronologically (3.0, 3.1 ... 7.0)
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    era_tags = eras[era]
    # Sort the tags within the era chronologically by date
    era_tags_sorted = sorted(era_tags, key=lambda x: x['date'])

    genesis = era_tags_sorted[0]
    sampled_tags.append(genesis)

    terminal = era_tags_sorted[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# --- Visual Verification Table ---
print("--- 🗺️ Phase 3: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    role = "Genesis Point (.0)" if tag_name.endswith('.0') or tag_name.endswith('.0.RELEASE') else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")
print("\n")

# --- Final Clean JSON Output ---
print("--- 🏁 Phase 3: Final JSON Manifest for spring-framework.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")
print("  ]\n}")

### ⛏️ Repository Tag Extraction & Validation: Apache Dubbo
**Archetype:** Microservices / RPC Framework

This script maps the architectural evolution of **Apache Dubbo**, a high-performance, Java-based open-source RPC framework. Because Dubbo handles complex network routing, service discovery, and load balancing, its architectural inflection points provide unique insights into the technical debt profile of heavily distributed systems.

**The Repository Quirk: The `dubbo-` Prefix & Incubation History**
Apache Dubbo has a unique development history, having transitioned from an internal Alibaba project to a top-level Apache Software Foundation project. Consequently, its tagging culture is mixed. Formal releases traditionally use a `dubbo-X.Y.Z` prefix, but earlier or transitional tags sometimes use `vX.Y.Z` or just pure `X.Y.Z`. Additionally, the repository is filled with `.preview` and `-beta` tags that must be rigorously filtered out.

To capture the true structural evolution of the framework, this script utilizes a three-phase extraction:
* **Phase 1 & 2: Strict Regex Allowlist:** Clones the repository and applies a targeted Regular Expression (`^(?:dubbo-|v)?(\d+)\.(\d+)\.(\d+)$`) to safely capture all valid GA formats while completely eradicating previews, betas, and release candidates.
* **Phase 3: Architectural Boundary Mapping:** Groups the surviving, clean GA releases into their `Major.Minor` architectural eras (e.g., `2.6`, `2.7`, `3.0`). It then extracts the chronologically first "Genesis Point" and the final "Terminal Point" of each era. This isolates the exact boundaries where major structural shifts and feature additions occurred, stripping away the noise of over 100 minor bug-fix patches.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/apache/dubbo.git"
REPO_DIR = "dubbo"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- Strict Allowlist Filter Logic ---
    match = re.match(r'^(?:dubbo-|v)?(\d+)\.(\d+)\.(\d+)$', tag)

    if not match:
        if re.search(r'\d+\.\d+\.\d+', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major, minor, patch = int(match.group(1)), int(match.group(2)), int(match.group(3))

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': f"{major}.{minor}", # Group by Major.Minor era
        'major': major,                # [FIX]: Save for mathematical sorting
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print(f"{'Clean GA Release':<30} | {'Release Date':<15} | {'SHA'}")
print("-" * 70)
for t in clean_ga_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    # [FIX]: Sort mathematically by SemVer, NOT by Git date
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# --- Visual Verification Table ---
print("--- 🗺️ Phase 3: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # [FIX]: Determine Genesis by its position in the era list, not just if it ends in .0
    role = "Genesis Point" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")
print("\n")

# --- Final Clean JSON Output ---
print("--- 🏁 Phase 3: Final JSON Manifest for dubbo.json ---")
print("{\n  \"versions\": [")

for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")

print("  ]\n}")

### ⛏️ Repository Tag Extraction & Validation: QuestDB
**Archetype:** High-Performance Time-Series Database

This script maps the evolutionary timeline of **QuestDB**. As a high-performance database optimized for zero-GC (garbage collection) Java and raw hardware efficiency, its technical debt profile is highly sensitive to core storage engine rewrites.

**The Repository Quirk: The Optional Patch Digit**
QuestDB maintains a very clean tagging culture, but with one mathematical quirk: developers frequently omit the third `.0` digit when tagging the start of a new Major or Minor era (e.g., tagging `1.0` instead of `1.0.0`, or `6.1` instead of `6.1.0`). A standard Semantic Versioning parser would discard these as invalid, accidentally deleting the true Genesis points of the architecture.

To capture these shifts precisely, this script utilizes a three-phase extraction:
* **Phase 1 & 2: Adaptive Regex Filtering:** Uses an advanced Regular Expression (`^(?:v)?(\d+)\.(\d+)(?:\.(\d+))?$`) that makes the third digit optional. It mathematically infers the missing `.0` to ensure true initial releases are captured while still safely discarding alphas, betas, and release candidates.
* **Phase 3: Inflection Threshold Mapping:** Groups the clean GA releases into their `Major.Minor` architectural eras. It extracts the "Genesis Point" (the inferred `.0` release introducing the new architecture) and the "Terminal Point" (the final patch representing the mature architecture) of each era, isolating the bounds of structural code decay.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/questdb/questdb.git"
REPO_DIR = "questdb"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- UPDATED ALLOWLIST LOGIC ---
    # Now allows X.Y.Z OR X.Y (makes the third digit optional)
    match = re.match(r'^(?:v)?(\d+)\.(\d+)(?:\.(\d+))?$', tag)

    if not match:
        if re.search(r'\d+\.\d+', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major = int(match.group(1))
    minor = int(match.group(2))
    # If the tag is just "1.0", treat the patch as 0
    patch = int(match.group(3)) if match.group(3) else 0

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': f"{major}.{minor}",
        'major': major,                # [FIX]: Required for mathematical sorting
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print(f"{'Clean GA Release':<30} | {'Release Date':<15} | {'SHA'}")
print("-" * 70)
for t in clean_ga_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    # [FIX]: Sort mathematically by SemVer, NOT by Git date to prevent backport time-travel
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

print("--- 🗺️ Phase 3: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # [FIX]: Dynamically determine Genesis by array position, not hardcoded patch numbers
    role = "Genesis Point" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")

print("\n--- 🏁 Final JSON Manifest for questdb.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")
print("  ]\n}")

### ⛏️ Repository Tag Extraction & Validation: Checkstyle
**Archetype:** Static Analysis / AST Tool

This script maps the evolutionary timeline of **Checkstyle**. Because Checkstyle is itself a static analysis tool designed to parse Abstract Syntax Trees (ASTs), analyzing its structural decay with a similar tool (PMD) provides a fascinating meta-analytical layer to the dataset.

**The Repository Quirk: "Minor is the new Patch" & The SVN Migration**
Checkstyle has two severe repository quirks that break standard extraction models:
1.  **The Monthly Cadence:** They follow a strict monthly release cycle, bumping the Minor version every month (e.g., 8.35, 8.36, 8.37) and reserving patches purely for emergencies. Grouping by `Major.Minor` here would incorrectly treat a single month of maintenance as a full "architectural era." We must instead group strictly by the **Major** version.
2.  **The Timestamp Corruption:** In June 2018, Checkstyle mass-migrated its legacy code from SVN to GitHub. Consequently, many v4 and v5 tags have the exact same Git creation date (`2018-06-02`), which breaks time-based chronological sorting.

To overcome these anomalies, this script executes a three-phase extraction:
* **Phase 1 & 2: Regex Filtering:** Applies an adaptive Regular Expression to capture standard and "missing third digit" tags while safely stripping out pre-releases.
* **Phase 3: Major Era Mathematical Mapping:** Groups releases strictly by their **Major** version era (v6, v7, v8...). To bypass the corrupted SVN timestamps, the script mathematically sorts the tags by their semantic version numbers (`X`, `Y`, `Z`) to flawlessly isolate the true Genesis and Terminal points of each major architecture.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/checkstyle/checkstyle.git"
REPO_DIR = "checkstyle"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- Strict Allowlist Filter Logic ---
    match = re.match(r'^(?:checkstyle-|v|release)?(\d+)[\._](\d+)(?:[\._](\d+))?$', tag)

    if not match:
        if re.search(r'\d+\.\d+(?:\.\d+)?', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major = int(match.group(1))
    minor = int(match.group(2))
    # If the tag is just "checkstyle-8.1", treat the patch as 0 mathematically
    patch = int(match.group(3)) if match.group(3) else 0

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': str(major), # Grouping purely by MAJOR era
        'major': major,
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)

# Iterate over the entire all_tags list instead of just the last 50
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: int(k))

for era in sorted_era_keys:
    # Sorting mathematically by version numbers to bypass the 2018 GitHub mass-migration date glitch
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# --- Visual Verification Table ---
print("--- 🗺️ Phase 3: Architectural Boundary Map (Major Eras) ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # If it is the absolute lowest minor/patch in this Major era, it's the Genesis
    role = "Genesis Point" if tag_obj == eras[str(tag_obj['major'])][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")

# --- Final Clean JSON Output ---
print("\n--- 🏁 Final JSON Manifest for checkstyle.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")
print("  ]\n}")